# EXE CLOUD · GPU
Pulsa **Entorno de ejecución → Ejecutar todo**. Si la celda de abajo dice que no hay GPU, elige **Cambiar tipo de entorno → GPU** y vuelve a ejecutar. La sesión la controlas tú: se cierra con el botón Desconectar de EXE CLOUD o parando la celda.

In [ ]:
!nvidia-smi || echo "No hay GPU NVIDIA en esta sesión"

In [ ]:
REPO = "llacskon33/exe-cloud-runner"
import getpass
TOKEN = getpass.getpass("Token de GitHub con permiso de escritura en el repositorio (solo para esta sesión): ")

# EXE CLOUD - servidor de tareas GPU para Google Colab.
# Solo hace conexiones de SALIDA hacia la API de GitHub: no abre puertos, ni SSH, ni túneles.
# Todos los datos de hardware salen de esta máquina (nvidia-smi, /proc/meminfo).
import os, re, sys, json, time, base64, subprocess, atexit, uuid
import urllib.request, urllib.error

API = os.environ.get("EXE_API", "https://api.github.com")
BRANCH = "gpu"
POLL = int(os.environ.get("EXE_POLL", "8"))
BEAT = int(os.environ.get("EXE_BEAT", "30"))
START = time.time()
SESSION = uuid.uuid4().hex[:8]
REPO = ""
TOKEN = ""
state = "checking"
done = failed = 0
current = None
LOG, ERRORS = [], []

TEST_CODE = """
import json, time
try:
    import torch
except Exception as e:
    print(json.dumps({"cuda": False, "error": "PyTorch no disponible: " + str(e)})); raise SystemExit(1)
if not torch.cuda.is_available():
    print(json.dumps({"cuda": False, "error": "CUDA no disponible en esta sesión"})); raise SystemExit(1)
d = torch.device("cuda"); n = 4096
a = torch.randn(n, n, device=d); b = torch.randn(n, n, device=d)
c = a @ b; torch.cuda.synchronize()
t = time.time()
for _ in range(20):
    c = a @ b
torch.cuda.synchronize(); dt = time.time() - t
print(json.dumps({"cuda": True, "device": torch.cuda.get_device_name(0), "gflops": round(20 * 2 * n ** 3 / dt / 1e9, 1), "seconds": round(dt, 3), "checksum": float(c.sum().cpu())}))
"""


def log(msg):
    line = time.strftime("%H:%M:%S") + " " + str(msg)
    print(line, flush=True)
    LOG.append(line)
    del LOG[:-15]


def err(msg):
    log("ERROR " + str(msg))
    ERRORS.append(time.strftime("%H:%M:%S") + " " + str(msg))
    del ERRORS[:-10]


def call(method, path, body=None):
    data = json.dumps(body).encode() if body is not None else None
    req = urllib.request.Request(API + path, data=data, method=method, headers={
        "Authorization": "Bearer " + TOKEN, "Accept": "application/vnd.github+json",
        "Content-Type": "application/json", "User-Agent": "exe-cloud-gpu"})
    try:
        with urllib.request.urlopen(req, timeout=30) as r:
            raw = r.read()
            return r.status, (json.loads(raw) if raw else None)
    except urllib.error.HTTPError as e:
        raw = e.read()
        try:
            return e.code, json.loads(raw)
        except Exception:
            return e.code, None


def get_file(path):
    c, d = call("GET", "/repos/%s/contents/%s?ref=%s" % (REPO, path, BRANCH))
    if c == 404:
        return None
    if c != 200:
        raise RuntimeError("GitHub %s al leer %s" % (c, path))
    return json.loads(base64.b64decode(d["content"]).decode())


def list_dir(path):
    c, d = call("GET", "/repos/%s/contents/%s?ref=%s" % (REPO, path, BRANCH))
    if c == 404:
        return []
    if c != 200:
        raise RuntimeError("GitHub %s al listar %s" % (c, path))
    return [x["name"] for x in d]


def put_file(path, obj, msg):
    for _ in range(3):
        c, d = call("GET", "/repos/%s/contents/%s?ref=%s" % (REPO, path, BRANCH))
        body = {"message": msg, "branch": BRANCH, "content": base64.b64encode(json.dumps(obj).encode()).decode()}
        if c == 200 and isinstance(d, dict) and d.get("sha"):
            body["sha"] = d["sha"]
        c, d = call("PUT", "/repos/%s/contents/%s" % (REPO, path), body)
        if c in (200, 201):
            return
        if c in (409, 422):
            time.sleep(1)
            continue
        raise RuntimeError("GitHub %s al escribir %s" % (c, path))
    raise RuntimeError("conflicto al escribir " + path)


def gpu_info():
    try:
        out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.used,memory.free,driver_version",
                              "--format=csv,noheader,nounits"], capture_output=True, text=True, timeout=15)
    except (FileNotFoundError, subprocess.TimeoutExpired):
        return None
    if out.returncode != 0 or not out.stdout.strip():
        return None
    n, t, u, f, dv = [x.strip() for x in out.stdout.strip().splitlines()[0].split(",")]
    return {"name": n, "vram_total_mb": int(float(t)), "vram_used_mb": int(float(u)),
            "vram_free_mb": int(float(f)), "driver": dv}


def ram_mb():
    try:
        m = {}
        for line in open("/proc/meminfo"):
            k, v = line.split(":")
            m[k] = int(v.split()[0]) // 1024
        return m.get("MemTotal"), m.get("MemAvailable")
    except Exception:
        return None, None


def beat():
    total, avail = ram_mb()
    put_file("status.json", {
        "state": state, "session": SESSION, "started": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime(START)),
        "updated": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        "gpu": gpu_info(), "cpu": os.cpu_count(), "ram_total_mb": total, "ram_avail_mb": avail,
        "python": sys.version.split()[0], "jobs_done": done, "jobs_failed": failed, "current_job": current,
        "errors": ERRORS[-10:], "log": LOG[-10:]}, "gpu status")


def run_job(jid, j):
    global state, current, done, failed
    state, current = "processing", jid
    log("Ejecutando tarea " + jid)
    try:
        beat()
    except Exception as e:
        err(e)
    t0 = time.time()
    src = TEST_CODE if j.get("type") == "test" else str(j.get("code", ""))
    path = "/tmp/job_%s.py" % jid
    open(path, "w").write(src)
    rc, so, se = None, "", ""
    try:
        p = subprocess.run([sys.executable, path], capture_output=True, text=True, timeout=int(j.get("timeout", 600)))
        rc, so, se = p.returncode, p.stdout, p.stderr
    except subprocess.TimeoutExpired:
        se = "Tiempo agotado"
    except Exception as e:
        se = str(e)
    ok = rc == 0
    done += 1
    failed += 0 if ok else 1
    put_file("results/%s.json" % jid, {"id": jid, "ok": ok, "returncode": rc, "stdout": so[-8000:], "stderr": se[-4000:],
                                       "seconds": round(time.time() - t0, 2), "finished": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())},
             "result " + jid)
    log("Tarea %s %s" % (jid, "terminada" if ok else "con error"))
    state, current = "ready", None


def main(repo, token):
    global REPO, TOKEN, state
    REPO, TOKEN = repo, token
    c, d = call("GET", "/repos/" + repo)
    if c != 200:
        print("No se pudo acceder al repositorio %s (GitHub %s). Revisa el token." % (repo, c))
        return
    if isinstance(d, dict) and d.get("permissions") and not d["permissions"].get("push"):
        print("El token no tiene permiso de escritura en el repositorio.")
        return
    log("Sesión %s iniciada" % SESSION)
    state = "checking"
    beat()
    g = gpu_info()
    log("GPU: " + (g["name"] if g else "ninguna detectada (Entorno de ejecución > Cambiar tipo > GPU)"))
    state = "ready"
    atexit.register(lambda: None)
    known = set(list_dir("results"))
    last = 0
    try:
        while True:
            try:
                cmd = get_file("command.json")
                if cmd and cmd.get("cmd") == "stop" and cmd.get("ts", 0) > START * 1000:
                    log("Orden de desconexión recibida")
                    break
                pending = sorted(n for n in list_dir("jobs") if n.endswith(".json") and n not in known)
                for n in pending:
                    jid = re.sub(r"[^A-Za-z0-9_-]", "", n[:-5])
                    j = get_file("jobs/" + n)
                    if j:
                        run_job(jid, j)
                    known.add(n)
                if pending or time.time() - last > BEAT:
                    beat()
                    last = time.time()
            except KeyboardInterrupt:
                raise
            except Exception as e:
                err(e)
            time.sleep(POLL)
    except KeyboardInterrupt:
        log("Detenido a mano")
    finally:
        state = "ended"
        try:
            beat()
        except Exception:
            pass
        print("Sesión finalizada.")


main(REPO, TOKEN)